In [ ]:
%matplotlib inline
import sys, importlib
sys.path.insert(0, '.')
import functions; importlib.reload(functions)
from functions import fetch_4h, detect, chart

In [ ]:
# ============================================================
# 설정
# ============================================================
TICKER = "^NDX"          # 종목 (^NDX=나스닥100, QQQ=ETF, NQ=F=선물)
FETCH_DAYS = 730         # 데이터 수집 기간 (일)
RSI_PERIOD = 14          # RSI 계산 기간
LB_LEFT = 5              # 피봇 좌측 확인 봉수
LB_RIGHT = 5             # 피봇 우측 확인 봉수 (확정 지연 = LB_RIGHT × 4h)
RANGE_LOWER = 5          # 피봇 간 최소 간격 (봉)
RANGE_UPPER = 60         # 피봇 간 최대 간격 (봉)
LOOKBACK = 2             # 이전 N개 피봇까지 비교 (1=직전만, 2=1개 건너뜀 허용)
CHART_BARS = 200         # 차트에 표시할 최근 봉 수

In [ ]:
# 데이터 수집 (1h → 장중 세션 4h)
df = fetch_4h(TICKER, FETCH_DAYS)
print(f"{len(df)}봉  |  {df.index[0].date()} ~ {df.index[-1].date()}  |  종가 ${df['close'].iloc[-1]:.2f}")

In [ ]:
# 다이버전스 탐지
signals, rsi, pl, ph = detect(df, RSI_PERIOD, LB_LEFT, LB_RIGHT, RANGE_LOWER, RANGE_UPPER, LOOKBACK)
print(f"피봇: Low {len(pl)} / High {len(ph)}  |  시그널: {len(signals)}개")
for s in signals[-10:]:
    print(f"  [{s.label:6s}] {df.index[s.idx].strftime('%Y-%m-%d %H:%M')} | ${s.price:.2f} | RSI {s.rsi:.1f}")

In [ ]:
# 차트
chart(df, rsi, signals, pl, ph, TICKER, bars=CHART_BARS)

In [ ]:
# 넓은 범위
chart(df, rsi, signals, pl, ph, TICKER, bars=400)

In [ ]:
# 시그널 테이블
import pandas as pd
rows = []
for s in signals:
    pct = (df['close'].iloc[-1] - df['close'].iloc[s.idx]) / df['close'].iloc[s.idx] * 100
    rows.append({"Type": s.label, "Date": df.index[s.idx].strftime('%Y-%m-%d %H:%M'),
                 "Price": f"${s.price:.2f}", "RSI": f"{s.rsi:.1f}", "Since": f"{pct:+.1f}%"})
pd.DataFrame(rows)